# Regenerate Plots — All Completed Models (154-Class)
**Dissertation: Real-Time ASL Recognition** | Owasu Brown | NU 2025–2026

Re-generates all visualisations from saved prediction/probability files.
No retraining or re-inference needed.

## Outputs per model (all saved to `results/end_point/`, existing files overwritten)
| File | Description |
|---|---|
| `{model}_top20_barplot.png` | Top-20 classes by F1, sorted bar chart |
| `{model}_performance_table.png` | Global metrics banner + Top-10 best + Bottom-10 worst classes |
| `{model}_f1_histogram.png` | Class count per F1 bin |
| `{model}_cumulative_f1.png` | % of classes above each F1 threshold |
| `{model}_f1_vs_support.png` | F1 vs number of test videos per class |
| `{model}_per_class_metrics.csv` | Full per-class metrics table |

## Data sources — 154-class experiment
| Model | Proba array | y_true source |
|---|---|---|
| InceptionTime | `inc_154/inc_154_proba_test.npy` | `data/npz/ASL_154_test_cif.npz` |
| Random Forest | `rf_154/test/rf_154_proba_test.npy` | `data/csv/ASL_154_test_rf.csv` |
| Logistic Regression | `lr_154/lr_154_proba_test.npy` | `data/csv/ASL_154_test_rf.csv` |
| CIF (50 trees) | `cif_154/cif_154_proba_test.npy` | `data/npz/ASL_154_test_cif.npz` |

Labels loaded from `data/meta/ASL_154_label_encoder.pkl`.

**Run order:** Cell 1 → 2 → 3 → 4 → then Cell 5 (all at once) or any individual cell.


In [1]:
# ── Cell 1: Install packages ──────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'joblib', '-q'
])
print('✅ Packages ready')


✅ Packages ready


In [2]:
# ── Cell 2: Mount Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# ── Cell 3: Paths & label encoder ────────────────────────────────────────
import os, sys, json, warnings
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings('ignore')

PROJECT_DIR = ('/content/drive/MyDrive/Consultant/Colab_Notebooks/'
               'Oberon_Dissertation_NU_25/OBrown_DIS9300_v2')
PROJECT_DIR = ('/content/drive/MyDrive/Consultant/Colab_Notebooks/'
               'Obrown_Dissertation_NU_25/OBrown_DIS9300_v2')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
END_DIR     = os.path.join(RESULTS_DIR, 'end_point')
os.makedirs(END_DIR, exist_ok=True)

# ── Label encoder ─────────────────────────────────────────────────────────
LE_CANDIDATES = [
    os.path.join(PROJECT_DIR, 'data', 'meta', 'ASL_154_label_encoder.pkl'),
    os.path.join(PROJECT_DIR, 'ASL_154_label_encoder.pkl'),
    os.path.join(PROJECT_DIR, 'data', 'ASL_154_label_encoder.pkl'),
]
LE_PATH = next((p for p in LE_CANDIDATES if os.path.exists(p)), None)
if LE_PATH is None:
    raise FileNotFoundError(
        'ASL_154_label_encoder.pkl not found. Searched:\n  ' +
        '\n  '.join(LE_CANDIDATES)
    )

le          = joblib.load(LE_PATH)
LABEL_NAMES = list(le.classes_)
N_CLASSES   = len(LABEL_NAMES)

print(f'✅ Label encoder : {N_CLASSES} classes  →  {LE_PATH}')
print(f'   First 5 : {LABEL_NAMES[:5]}')
print(f'   Last  5 : {LABEL_NAMES[-5:]}')
print(f'✅ Output dir    : {END_DIR}')


✅ Label encoder : 154 classes  →  /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/data/meta/ASL_154_label_encoder.pkl
   First 5 : [np.str_('accident'), np.str_('africa'), np.str_('apple'), np.str_('argue'), np.str_('baby')]
   Last  5 : [np.str_('wrong'), np.str_('year'), np.str_('yes'), np.str_('yesterday'), np.str_('you')]
✅ Output dir    : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point


In [4]:
# ── Cell 4: Plot functions ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix
)

# ── Colour palette ────────────────────────────────────────────────────────
_BLUE  = '#2E75B6'
_TEAL  = '#2A9D8F'
_RED   = '#E63946'
_DARK  = '#1F3864'
_GREY  = '#6C757D'
_GOLD  = '#F4A261'
_GREEN = '#43AA8B'

plt.rcParams.update({
    'figure.dpi'     : 150,
    'savefig.dpi'    : 150,
    'font.size'      : 10,
    'axes.titlesize' : 11,
    'axes.labelsize' : 10,
})


# ══════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════════

def load_proba_and_labels(slot):
    """
    Load y_true (int), y_pred (int), y_proba (float32) from a model slot.
    slot keys: npy_path, y_true_src ('npz'|'csv'), y_src_path
    """
    npy_path   = slot['npy_path']
    y_true_src = slot['y_true_src']
    y_src_path = slot['y_src_path']

    if not os.path.exists(npy_path):
        raise FileNotFoundError(f'Proba array not found:\n  {npy_path}')
    if not os.path.exists(y_src_path):
        raise FileNotFoundError(f'y_true source not found:\n  {y_src_path}')

    y_proba = np.load(npy_path, allow_pickle=True).astype(np.float32)

    if y_true_src == 'npz':
        d = np.load(y_src_path, allow_pickle=True)
        for key in ['y', 'y_test', 'labels', 'label']:
            if key in d:
                y_true = d[key].astype(int)
                break
        else:
            raise KeyError(f'No label key in {y_src_path}. Keys: {list(d.keys())}')
    else:  # csv
        df  = pd.read_csv(y_src_path)
        col = next((c for c in df.columns
                    if c.lower() in ['label', 'gloss', 'y_true', 'class']), None)
        if col is None:
            raise KeyError(f'No label column in {y_src_path}. Cols: {list(df.columns)}')
        raw    = df[col].values
        lmap   = {v: i for i, v in enumerate(np.unique(raw))}
        y_true = np.array([lmap[v] for v in raw])

    n       = min(len(y_true), len(y_proba))
    y_true  = y_true[:n]
    y_proba = y_proba[:n]
    y_pred  = np.argmax(y_proba, axis=1)
    return y_true, y_pred, y_proba


def build_per_class_df(y_true, y_pred, label_names):
    """Build a per-class metrics DataFrame with columns:
       label, f1, precision, recall, support.
    """
    from sklearn.metrics import precision_recall_fscore_support
    n_cls = len(label_names)
    labels_present = sorted(set(y_true) | set(y_pred))

    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred,
        labels=list(range(n_cls)),
        average=None,
        zero_division=0
    )
    rows = []
    for i in range(n_cls):
        rows.append({
            'label'    : label_names[i],
            'f1'       : round(float(f1[i]),        4),
            'precision': round(float(prec[i]),      4),
            'recall'   : round(float(rec[i]),       4),
            'support'  : int(sup[i]),
        })
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════
# PLOT 1 — TOP-20 BAR CHART
# ══════════════════════════════════════════════════════════════════════════

def plot_top20_barplot(per_class_df, model_name, dataset_label):
    """Top-20 classes by F1, horizontal bar chart. Saved to END_DIR."""
    top20 = (per_class_df[per_class_df['support'] > 0]
             .sort_values('f1', ascending=False)
             .head(20))

    fig, ax = plt.subplots(figsize=(11, 7))
    colours = [_GREEN if v >= 0.8 else _BLUE if v >= 0.5 else _RED
               for v in top20['f1']]
    bars = ax.barh(top20['label'], top20['f1'],
                   color=colours, edgecolor='white', linewidth=0.5)
    ax.set_xlim(0, 1.08)
    ax.set_xlabel('F1 Score', fontsize=11)
    ax.set_title(
        f'Top-20 Classes by F1 Score\n'
        f'{model_name}  |  {dataset_label}',
        fontsize=12, color=_DARK, fontweight='bold'
    )
    ax.invert_yaxis()
    ax.axvline(0.5, color=_GREY,  linestyle='--', linewidth=1, alpha=0.6)
    ax.axvline(0.8, color=_GREEN, linestyle='--', linewidth=1, alpha=0.6)
    ax.text(0.505, len(top20)-0.5, 'F1=0.50', color=_GREY,  fontsize=7.5, va='top')
    ax.text(0.805, len(top20)-0.5, 'F1=0.80', color=_GREEN, fontsize=7.5, va='top')

    for bar, (_, row) in zip(bars, top20.iterrows()):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{row["f1"]:.3f}  (n={row["support"]})',
                va='center', ha='left', fontsize=8, color=_DARK)

    legend_elements = [
        mpatches.Patch(facecolor=_GREEN, label='F1 ≥ 0.80'),
        mpatches.Patch(facecolor=_BLUE,  label='0.50 ≤ F1 < 0.80'),
        mpatches.Patch(facecolor=_RED,   label='F1 < 0.50'),
    ]
    ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()

    path = os.path.join(END_DIR, f'{model_name}_top20_barplot.png')
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print(f'  ✅ top20_barplot        : {os.path.basename(path)}')


# ══════════════════════════════════════════════════════════════════════════
# PLOT 2 — PERFORMANCE TABLE
# ══════════════════════════════════════════════════════════════════════════

def plot_performance_table(y_true, y_pred, per_class_df, model_name, dataset_label):
    """
    Global metrics banner  +  Top-10 best classes  +  Bottom-10 worst classes.
    Saved to END_DIR.
    """
    acc      = accuracy_score(y_true, y_pred)
    f1_mac   = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    f1_wt    = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    prec_mac = precision_score(y_true, y_pred, average='macro',    zero_division=0)
    rec_mac  = recall_score(y_true, y_pred,    average='macro',    zero_division=0)
    n_test   = len(y_true)

    with_support = per_class_df[per_class_df['support'] > 0]
    top10    = with_support.sort_values('f1', ascending=False).head(10)
    bot10    = with_support.sort_values('f1', ascending=True ).head(10)

    fig = plt.figure(figsize=(15, 12))
    gs  = gridspec.GridSpec(3, 1, height_ratios=[1, 4, 4], hspace=0.45)

    # ── Banner: global metrics ────────────────────────────────────────────
    ax0 = fig.add_subplot(gs[0])
    ax0.axis('off')
    metrics_text = (
        f'Accuracy: {acc*100:.2f}%     '
        f'Macro F1: {f1_mac:.4f}     '
        f'Weighted F1: {f1_wt:.4f}     '
        f'Macro Precision: {prec_mac:.4f}     '
        f'Macro Recall: {rec_mac:.4f}     '
        f'Test samples: {n_test:,}'
    )
    ax0.text(0.5, 0.65, f'{model_name}  |  {dataset_label}',
             ha='center', va='center', fontsize=13,
             color=_DARK, fontweight='bold', transform=ax0.transAxes)
    ax0.text(0.5, 0.2, metrics_text,
             ha='center', va='center', fontsize=10,
             color='#333333', transform=ax0.transAxes,
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#EEF2F7',
                       edgecolor=_DARK, linewidth=1.2))

    def _draw_table(ax, df, title, header_colour):
        ax.axis('off')
        ax.set_title(title, fontsize=11, color=_DARK, fontweight='bold', pad=8)
        col_labels = ['Rank', 'Class Label', 'F1', 'Precision', 'Recall', 'Support']
        table_data = [
            [str(rank),
             row['label'],
             f'{row["f1"]:.4f}',
             f'{row["precision"]:.4f}',
             f'{row["recall"]:.4f}',
             str(row['support'])]
            for rank, (_, row) in enumerate(df.iterrows(), 1)
        ]
        tbl = ax.table(cellText=table_data, colLabels=col_labels,
                       cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        for (r, c), cell in tbl.get_celld().items():
            if r == 0:
                cell.set_facecolor(header_colour)
                cell.set_text_props(color='white', fontweight='bold')
            elif r % 2 == 0:
                cell.set_facecolor('#F5F5F5')
            else:
                cell.set_facecolor('#FFFFFF')
            cell.set_edgecolor('#DDDDDD')

    _draw_table(fig.add_subplot(gs[1]), top10,
                'Top-10 Best Performing Classes', _DARK)
    _draw_table(fig.add_subplot(gs[2]), bot10,
                'Bottom-10 Worst Performing Classes', _RED)

    path = os.path.join(END_DIR, f'{model_name}_performance_table.png')
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print(f'  ✅ performance_table    : {os.path.basename(path)}')


# ══════════════════════════════════════════════════════════════════════════
# PLOT 3 — F1 HISTOGRAM
# ══════════════════════════════════════════════════════════════════════════

def plot_f1_histogram(per_class_df, model_name, dataset_label):
    """Distribution of F1 scores across all classes. Saved to END_DIR."""
    f1_vals = per_class_df[per_class_df['support'] > 0]['f1'].values
    bins    = np.arange(0, 1.05, 0.05)

    fig, ax = plt.subplots(figsize=(10, 5))
    n, _, patches = ax.hist(f1_vals, bins=bins, edgecolor='white', linewidth=0.6)

    for patch, left in zip(patches, bins[:-1]):
        patch.set_facecolor(_GREEN if left >= 0.80 else
                            _BLUE  if left >= 0.50 else _RED)

    ax.axvline(f1_vals.mean(),   color=_DARK, linestyle='--', linewidth=1.5,
               label=f'Mean F1 = {f1_vals.mean():.3f}')
    ax.axvline(np.median(f1_vals), color=_GOLD, linestyle=':',  linewidth=1.5,
               label=f'Median F1 = {np.median(f1_vals):.3f}')

    n_zero = (f1_vals == 0).sum()
    ax.set_xlabel('F1 Score', fontsize=11)
    ax.set_ylabel('Number of Classes', fontsize=11)
    ax.set_title(
        f'F1 Score Distribution — {model_name}  |  {dataset_label}\n'
        f'{len(f1_vals)} classes with test support  |  '
        f'{n_zero} classes at F1=0  |  '
        f'Mean={f1_vals.mean():.3f}  Median={np.median(f1_vals):.3f}',
        fontsize=10, color=_DARK
    )
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    legend_elements = [
        mpatches.Patch(facecolor=_GREEN, label='F1 ≥ 0.80'),
        mpatches.Patch(facecolor=_BLUE,  label='0.50 ≤ F1 < 0.80'),
        mpatches.Patch(facecolor=_RED,   label='F1 < 0.50'),
    ]
    ax.legend(handles=legend_elements + ax.get_legend_handles_labels()[0][-2:],
              fontsize=9)
    plt.tight_layout()

    path = os.path.join(END_DIR, f'{model_name}_f1_histogram.png')
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print(f'  ✅ f1_histogram         : {os.path.basename(path)}')


# ══════════════════════════════════════════════════════════════════════════
# PLOT 4 — CUMULATIVE F1
# ══════════════════════════════════════════════════════════════════════════

def plot_cumulative_f1(per_class_df, model_name, dataset_label):
    """% of classes that achieve at least F1=t, for t in [0,1]. Saved to END_DIR."""
    f1_vals   = np.sort(per_class_df[per_class_df['support'] > 0]['f1'].values)
    n_classes = len(f1_vals)
    thresholds = np.linspace(0, 1, 200)
    pct_above  = [(f1_vals >= t).mean() * 100 for t in thresholds]

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(thresholds, pct_above, color=_BLUE, linewidth=2.5)
    ax.fill_between(thresholds, pct_above, alpha=0.12, color=_BLUE)

    for threshold, colour, label in [
        (0.50, _GREY,  '50%'),
        (0.80, _GREEN, '80%'),
    ]:
        pct = (f1_vals >= threshold).mean() * 100
        ax.axvline(threshold, color=colour, linestyle='--', linewidth=1.2)
        ax.axhline(pct,       color=colour, linestyle=':',  linewidth=1.0)
        ax.scatter([threshold], [pct], color=colour, s=60, zorder=5)
        ax.annotate(f'F1≥{threshold:.2f} → {pct:.1f}% of classes',
                    (threshold, pct), xytext=(threshold+0.03, pct+3),
                    fontsize=8.5, color=colour)

    ax.set_xlabel('F1 Threshold', fontsize=11)
    ax.set_ylabel('% of Classes Achieving F1 ≥ Threshold', fontsize=11)
    ax.set_title(
        f'Cumulative F1 Performance — {model_name}  |  {dataset_label}\n'
        f'{n_classes} classes with test support',
        fontsize=11, color=_DARK
    )
    ax.set_xlim(0, 1); ax.set_ylim(0, 105)
    ax.grid(alpha=0.3)
    plt.tight_layout()

    path = os.path.join(END_DIR, f'{model_name}_cumulative_f1.png')
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print(f'  ✅ cumulative_f1        : {os.path.basename(path)}')


# ══════════════════════════════════════════════════════════════════════════
# PLOT 5 — F1 VS SUPPORT
# ══════════════════════════════════════════════════════════════════════════

def plot_f1_vs_support(per_class_df, model_name, dataset_label):
    """Scatter of F1 vs test-set support per class. Saved to END_DIR."""
    df = per_class_df[per_class_df['support'] > 0].copy()

    fig, ax = plt.subplots(figsize=(10, 6))
    colours = [_GREEN if v >= 0.8 else _BLUE if v >= 0.5 else _RED
               for v in df['f1']]
    ax.scatter(df['support'], df['f1'], c=colours, alpha=0.65,
               s=35, edgecolors='white', linewidths=0.4)

    # Annotate top-5 and bottom-5 by F1
    for _, row in df.nlargest(5, 'f1').iterrows():
        ax.annotate(row['label'], (row['support'], row['f1']),
                    xytext=(4, 2), textcoords='offset points',
                    fontsize=7, color=_GREEN)
    for _, row in df.nsmallest(5, 'f1').iterrows():
        ax.annotate(row['label'], (row['support'], row['f1']),
                    xytext=(4, -8), textcoords='offset points',
                    fontsize=7, color=_RED)

    ax.axhline(0.5, color=_GREY,  linestyle='--', linewidth=1, alpha=0.7)
    ax.axhline(0.8, color=_GREEN, linestyle='--', linewidth=1, alpha=0.7)
    ax.set_xlabel('Test Support (number of test videos per class)', fontsize=11)
    ax.set_ylabel('F1 Score', fontsize=11)
    ax.set_title(
        f'F1 Score vs Test Support — {model_name}  |  {dataset_label}\n'
        f'{len(df)} classes  |  '
        f'Spearman r = {df["f1"].corr(df["support"], method="spearman"):.3f}',
        fontsize=11, color=_DARK
    )

    legend_elements = [
        mpatches.Patch(facecolor=_GREEN, label='F1 ≥ 0.80'),
        mpatches.Patch(facecolor=_BLUE,  label='0.50 ≤ F1 < 0.80'),
        mpatches.Patch(facecolor=_RED,   label='F1 < 0.50'),
    ]
    ax.legend(handles=legend_elements, fontsize=9)
    ax.grid(alpha=0.25)
    plt.tight_layout()

    path = os.path.join(END_DIR, f'{model_name}_f1_vs_support.png')
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print(f'  ✅ f1_vs_support        : {os.path.basename(path)}')


# ══════════════════════════════════════════════════════════════════════════
# MASTER RUNNER
# ══════════════════════════════════════════════════════════════════════════

def regenerate_plots(slot, model_name, dataset_label='154-class'):
    """
    Full regeneration pipeline for one model.
    Loads data, builds per-class metrics, generates all 5 plots + CSV.
    All outputs saved to results/end_point/. Existing files overwritten.
    """
    sep = '=' * 66
    print(f'\n{sep}')
    print(f'  {model_name}  |  {dataset_label}')
    print(sep)

    # Verify files upfront
    for key, path in [('npy_path', slot['npy_path']),
                      ('y_src_path', slot['y_src_path'])]:
        if not os.path.exists(path):
            print(f'  ⬜ SKIP — {key} not found:\n     {path}')
            return

    y_true, y_pred, y_proba = load_proba_and_labels(slot)
    print(f'  Loaded: {len(y_true):,} samples  |  '
          f'{y_proba.shape[1]} output classes  |  '
          f'Accuracy = {(y_true == y_pred).mean()*100:.2f}%')

    per_class_df = build_per_class_df(y_true, y_pred, LABEL_NAMES)

    # Save per-class CSV (overwrite)
    csv_path = os.path.join(END_DIR, f'{model_name}_per_class_metrics.csv')
    per_class_df.to_csv(csv_path, index=False)
    print(f'  ✅ per_class_metrics    : {os.path.basename(csv_path)}')

    # Generate all plots
    plot_top20_barplot(per_class_df, model_name, dataset_label)
    plot_performance_table(y_true, y_pred, per_class_df, model_name, dataset_label)
    plot_f1_histogram(per_class_df, model_name, dataset_label)
    plot_cumulative_f1(per_class_df, model_name, dataset_label)
    plot_f1_vs_support(per_class_df, model_name, dataset_label)

    print(f'  ✅ Done → {END_DIR}')


print('✅ All plot functions defined.')


✅ All plot functions defined.


In [5]:
# ── Cell 5: Model registry — 154-class experiment ────────────────────────
P = PROJECT_DIR
R = RESULTS_DIR

MODELS_154 = [
    dict(
        name       = 'InceptionTime_154',
        label      = '154-class',
        npy_path   = f'{R}/inc_154/inc_154_proba_test.npy',
        y_true_src = 'npz',
        y_src_path = f'{P}/data/npz/ASL_154_test_cif.npz',
    ),
    dict(
        name       = 'RandomForest_154',
        label      = '154-class',
        npy_path   = f'{R}/rf_154/rf_154_proba_test.npy',
        y_true_src = 'csv',
        y_src_path = f'{P}/data/csv/ASL_154_test_rf.csv',
    ),
    dict(
        name       = 'LogisticRegression_154',
        label      = '154-class',
        npy_path   = f'{R}/lr_154/lr_154_proba_test.npy',
        y_true_src = 'csv',
        y_src_path = f'{P}/data/csv/ASL_154_test_rf.csv',
    ),
    dict(
        name       = 'CIF_50trees_154',
        label      = '154-class',
        npy_path   = f'{R}/cif_154/cif_154_proba_test.npy',
        y_true_src = 'npz',
        y_src_path = f'{P}/data/npz/ASL_154_test_cif.npz',
    ),
]

print('Registry — 154-class models:')
print(f'  {"Model":<28}  {"Proba .npy":<6}  {"y_true src":<6}')
print('  ' + '-' * 56)
all_ok = True
for s in MODELS_154:
    npy_ok = '\u2705' if os.path.exists(s['npy_path']) else '\u274c MISSING'
    src_ok = '\u2705' if os.path.exists(s['y_src_path']) else '\u274c MISSING'
    if '\u274c' in npy_ok or '\u274c' in src_ok:
        all_ok = False
    print(f'  {s["name"]:<28}  npy:{npy_ok}  src:{src_ok}')
print()
if all_ok:
    print('\u2705 All files found — safe to run Cell 6.')
else:
    print('\u26a0\ufe0f  Some files missing — check paths above before running Cell 6.')


Registry — 154-class models:
  Model                         Proba .npy  y_true src
  --------------------------------------------------------
  InceptionTime_154             npy:✅  src:✅
  RandomForest_154              npy:✅  src:✅
  LogisticRegression_154        npy:✅  src:✅
  CIF_50trees_154               npy:✅  src:✅

✅ All files found — safe to run Cell 6.


In [6]:
# ── Cell 6: Regenerate ALL models ────────────────────────────────────────
# Produces 6 files per model (5 PNGs + 1 CSV) in results/end_point/.
# Existing files are overwritten silently.

for slot in MODELS_154:
    regenerate_plots(slot, model_name=slot['name'], dataset_label=slot['label'])

print('\n' + '=' * 66)
print('  ALL MODELS COMPLETE')
print(f'  Outputs saved to: {END_DIR}')
print('=' * 66)



  InceptionTime_154  |  154-class
  Loaded: 309 samples  |  154 output classes  |  Accuracy = 35.60%
  ✅ per_class_metrics    : InceptionTime_154_per_class_metrics.csv
  ✅ top20_barplot        : InceptionTime_154_top20_barplot.png
  ✅ performance_table    : InceptionTime_154_performance_table.png
  ✅ f1_histogram         : InceptionTime_154_f1_histogram.png
  ✅ cumulative_f1        : InceptionTime_154_cumulative_f1.png
  ✅ f1_vs_support        : InceptionTime_154_f1_vs_support.png
  ✅ Done → /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point

  RandomForest_154  |  154-class
  Loaded: 309 samples  |  154 output classes  |  Accuracy = 20.39%
  ✅ per_class_metrics    : RandomForest_154_per_class_metrics.csv
  ✅ top20_barplot        : RandomForest_154_top20_barplot.png
  ✅ performance_table    : RandomForest_154_performance_table.png
  ✅ f1_histogram         : RandomForest_154_f1_histogram.png
  ✅ cumulative_f1        : RandomF

---
## Run individual models
Use the cells below to regenerate one model at a time.


### InceptionTime_154

In [7]:
# ── InceptionTime_154 ──────────────────────────────────────────────────────
regenerate_plots(
    slot         = MODELS_154[0],
    model_name   = MODELS_154[0]['name'],
    dataset_label= MODELS_154[0]['label'],
)



  InceptionTime_154  |  154-class
  Loaded: 309 samples  |  154 output classes  |  Accuracy = 35.60%
  ✅ per_class_metrics    : InceptionTime_154_per_class_metrics.csv
  ✅ top20_barplot        : InceptionTime_154_top20_barplot.png
  ✅ performance_table    : InceptionTime_154_performance_table.png
  ✅ f1_histogram         : InceptionTime_154_f1_histogram.png
  ✅ cumulative_f1        : InceptionTime_154_cumulative_f1.png
  ✅ f1_vs_support        : InceptionTime_154_f1_vs_support.png
  ✅ Done → /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point


### RandomForest_154

In [8]:
# ── RandomForest_154 ──────────────────────────────────────────────────────
regenerate_plots(
    slot         = MODELS_154[1],
    model_name   = MODELS_154[1]['name'],
    dataset_label= MODELS_154[1]['label'],
)



  RandomForest_154  |  154-class
  Loaded: 309 samples  |  154 output classes  |  Accuracy = 20.39%
  ✅ per_class_metrics    : RandomForest_154_per_class_metrics.csv
  ✅ top20_barplot        : RandomForest_154_top20_barplot.png
  ✅ performance_table    : RandomForest_154_performance_table.png
  ✅ f1_histogram         : RandomForest_154_f1_histogram.png
  ✅ cumulative_f1        : RandomForest_154_cumulative_f1.png
  ✅ f1_vs_support        : RandomForest_154_f1_vs_support.png
  ✅ Done → /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point


### LogisticRegression_154

In [9]:
# ── LogisticRegression_154 ──────────────────────────────────────────────────────
regenerate_plots(
    slot         = MODELS_154[2],
    model_name   = MODELS_154[2]['name'],
    dataset_label= MODELS_154[2]['label'],
)



  LogisticRegression_154  |  154-class
  Loaded: 309 samples  |  154 output classes  |  Accuracy = 24.60%
  ✅ per_class_metrics    : LogisticRegression_154_per_class_metrics.csv
  ✅ top20_barplot        : LogisticRegression_154_top20_barplot.png
  ✅ performance_table    : LogisticRegression_154_performance_table.png
  ✅ f1_histogram         : LogisticRegression_154_f1_histogram.png
  ✅ cumulative_f1        : LogisticRegression_154_cumulative_f1.png
  ✅ f1_vs_support        : LogisticRegression_154_f1_vs_support.png
  ✅ Done → /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point


### CIF_50trees_154

In [10]:
# ── CIF_50trees_154 ──────────────────────────────────────────────────────
regenerate_plots(
    slot         = MODELS_154[3],
    model_name   = MODELS_154[3]['name'],
    dataset_label= MODELS_154[3]['label'],
)



  CIF_50trees_154  |  154-class
  Loaded: 309 samples  |  154 output classes  |  Accuracy = 5.83%
  ✅ per_class_metrics    : CIF_50trees_154_per_class_metrics.csv
  ✅ top20_barplot        : CIF_50trees_154_top20_barplot.png
  ✅ performance_table    : CIF_50trees_154_performance_table.png
  ✅ f1_histogram         : CIF_50trees_154_f1_histogram.png
  ✅ cumulative_f1        : CIF_50trees_154_cumulative_f1.png
  ✅ f1_vs_support        : CIF_50trees_154_f1_vs_support.png
  ✅ Done → /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point
